# 🧠 การฝังตัวคุณลักษณะ (Feature Embeddings): เมทริกซ์ความคล้ายคลึงและการระบุตัวตนวัตถุซ้ำ (Object Re-ID)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Feature Embeddings**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายคณิตศาสตร์ของความคล้ายคลึงโคไซน์ (Cosine Similarity) และระยะทางยูคลิเดียน (Euclidean Distance)
2. พัฒนาการวัดทั้งสองแบบจากศูนย์ด้วย NumPy
3. จำลอง **สถานการณ์การติดตามวัตถุหลายชิ้น (Re-ID)** โดยใช้เวกเตอร์ฝังตัว (embedding vectors) ขนาด 128 มิติ
4. คำนวณเมทริกซ์ความคล้ายคลึงแบบจับคู่ (pairwise similarity matrix) และแสดงผลในรูปแบบแผนภูมิความร้อน (heatmap)
5. สาธิตวิธีที่อัลกอริทึมการติดตามใช้วัดค่าความเหมือนร่วมกับเกณฑ์ความคล้ายคลึง (similarity thresholds) ในการเชื่อมโยงวัตถุข้ามเฟรมภาพ
6. เชื่อมโยงเวกเตอร์ฝังตัวคุณลักษณะเข้ากับโมเดลการติดตามของ YOLO (เช่น DeepSORT, BoT-SORT)

เรามาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างตัวชี้วัดความคล้ายคลึงจากศูนย์

เรามาเขียนฟังก์ชันสำหรับ:
-   **Cosine Similarity:** $\frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$
-   **Euclidean Distance ($L2$ Norm):** $\sqrt{\sum (u_i - v_i)^2}$

In [ ]:
def cosine_similarity(u, v):
    dot_product = np.dot(u, v)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0.0 or norm_v == 0.0:
        return 0.0
    return dot_product / (norm_u * norm_v)

def euclidean_distance(u, v):
    return np.linalg.norm(u - v)

## 2. การจำลองเวกเตอร์ระบุตัวตนวัตถุซ้ำ (Re-Identification Vectors)

เราจะสร้างเวกเตอร์ฝังตัว 4 ตัวที่มีขนาดมิติเท่ากับ 128:
1.  `v1_f1`: Valve 1 detected in Frame 1.
2.  `v1_f2`: Same Valve 1 detected in Frame 2 (with noise).
3.  `v2_f1`: Valve 2 detected in Frame 1 (different valve).
4.  `f1_f1`: Flange 1 detected in Frame 1 (different category).

In [ ]:
v1_f1 = np.random.rand(128)
v2_f1 = np.random.rand(128)
f1_f1 = np.random.rand(128)

v1_f2 = v1_f1 + np.random.normal(0, 0.08, 128)

v1_f1 /= np.linalg.norm(v1_f1)
v1_f2 /= np.linalg.norm(v1_f2)
v2_f1 /= np.linalg.norm(v2_f1)
f1_f1 /= np.linalg.norm(f1_f1)

## 3. การคำนวณแผนภูมิความร้อนของความคล้ายคลึง

เรามาคำนวณค่าความคล้ายคลึงโคไซน์แบบจับคู่ระหว่างเวกเตอร์เหล่านี้ และนำมาพล็อตกราฟแผนภูมิความร้อนกันครับ

In [ ]:
embeddings = [v1_f1, v1_f2, v2_f1, f1_f1]
labels = ["Valve 1 (Frame 1)", "Valve 1 (Frame 2)", "Valve 2 (Frame 1)", "Flange 1 (Frame 1)"]

sim_matrix = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        sim_matrix[i, j] = cosine_similarity(embeddings[i], embeddings[j])

plt.figure(figsize=(8, 6))
sns.heatmap(sim_matrix, annot=True, xticklabels=labels, yticklabels=labels, cmap='viridis', vmin=0.5, vmax=1.0)
plt.title('Pairwise Cosine Similarity Matrix')
plt.xticks(rotation=15, ha='right')
plt.yticks(rotation=0)
plt.show()

ดูแผนภูมิความร้อนสิครับ:
- **ความคล้ายคลึงระหว่าง Valve 1 (เฟรม 1) กับ Valve 1 (เฟรม 2):** สูงมากอย่างชัดเจน ($\approx 0.96$)
- **ความคล้ายคลึงระหว่าง Valve 1 กับ Valve 2 (ต่างวาล์วกัน):** ค่อนข้างต่ำ ($\approx 0.76$)
- **ความคล้ายคลึงระหว่าง Valve 1 กับ Flange 1 (วัตถุคนละประเภทกัน):** ต่ำมาก ($\approx 0.69$)

การใช้เกณฑ์การตัดสินความคล้ายคลึง (เช่น $T = 0.90$) จะช่วยให้ตัวติดตามวัตถุสามารถระบุได้อย่างมั่นใจว่า Valve 1 ในเฟรมที่ 2 คือวัตถุชิ้นเดิมในประวัติการเดินทาง ช่วยรักษา ID ของวัตถุให้ถูกต้องแม้จะเกิดการบดบังบางส่วนหรือเกิดการเคลื่อนย้ายตำแหน่งไป!

## 4. การเปรียบเทียบระหว่างระยะทางเชิงพื้นที่กับความคล้ายคลึง

เรามาเปรียบเทียบความแตกต่างระหว่างความคล้ายคลึงโคไซน์กับระยะทางยูคลิเดียน ($L2$ Distance) กันครับ

In [ ]:
print("--- Comparison Metrics (Valve 1 vs. Others) ---")
print(f"Same Valve (v1_f1 vs. v1_f2) -> Cosine Similarity: {cosine_similarity(v1_f1, v1_f2):.4f} | L2 Distance: {euclidean_distance(v1_f1, v1_f2):.4f}")
print(f"Diff Valve (v1_f1 vs. v2_f1) -> Cosine Similarity: {cosine_similarity(v1_f1, v2_f1):.4f} | L2 Distance: {euclidean_distance(v1_f1, v2_f1):.4f}")
print(f"Diff Class (v1_f1 vs. f1_f1) -> Cosine Similarity: {cosine_similarity(v1_f1, f1_f1):.4f} | L2 Distance: {euclidean_distance(v1_f1, f1_f1):.4f}")

สังเกตผลที่ได้ครับ:
- สำหรับเวกเตอร์ที่เหมือนกันทุกประการ ค่าความคล้ายคลึงโคไซน์จะมีค่าเท่ากับ $1.0$ และระยะทาง L2 จะมีค่าเท่ากับ $0.0$
- เมื่อเวกเตอร์มีความคล้ายคลึงกันลดลง ค่าโคไซน์จะหดตัวเล็กลงในขณะที่ระยะทาง L2 จะขยายใหญ่ขึ้น แสดงให้เห็นว่าทั้งสองตัวชี้วัดมีความสัมพันธ์ผกผันต่อกัน

## 💡 การเชื่อมโยงกับกระบวนการติดตามวัตถุของ YOLO (YOLO Tracking Pipelines)
*   **การเชื่อมโยงวัตถุ (Object Association):** ในตัวติดตามวัตถุอย่าง **DeepSORT** ตัวตรวจจับ YOLO จะทำการระบุตำแหน่งวัตถุ จากนั้น Re-ID CNN จะดึงเวกเตอร์ฝังตัวคุณลักษณะขนาดกะทัดรัดสำหรับแต่ละกรอบวัตถุที่ตรวจพบ
*   **อัลกอริทึมฮังการี (Hungarian Algorithm):** ตัวติดตามจะคำนวณเมทริกซ์ความคล้ายคลึงโคไซน์ระหว่างเวกเตอร์การติดตามที่มีอยู่กับภาพตรวจพบใหม่ จากนั้นจะแก้ปัญหาการจับคู่ตำแหน่งพิกัดโดยใช้อัลกอริทึมฮังการี วิธีนี้ช่วยป้องกันไม่ให้ ID ของตัวติดตามเกิดการสลับกัน (Track ID switching) เมื่อวัตถุเคลื่อนที่ตัดกันหรือถูกบดบังชั่วคราว